# Lab 04 error analysis solution

Make sure to use a GPU and have access to internet connection in the Kaggle notebook:

1. On the three dots on the top left, select "Notebook Options" and then "Accelerator" to choose the GPU P100, and select "Variables & Files" under Persistence. **Note that Kaggle allows 30h per week per user of accelerated computing. Plan your work accordingly. It takes time to load the data and you may experience unavailability of GPUs or Session Errors.**
1. Make sure your Kaggle account is phone verified by clicking "Get phone verified" in the left sidebar under "Notebook options" and following the steps (this step is required to switch on the internet connection needed to install packages). 
1. After phone verification, the full settings menu should be visible. Toggle the "Internet" switch.

More visualizations of the process to connect the notebook to the internet are provided [here](https://stackoverflow.com/questions/68142524/cannot-access-internet-on-kaggle-notebook)


## Requirements:

Regarding the data:

If you haven't uploaded the ECTI2021 dataset to Kaggle yet follows these instructions: 

1. Downloading the tiles of the ecti2021 here: [ecti2021.zip](https://www.dropbox.com/scl/fi/tuvroadxyummourvx6cmz/ecti2021.zip?rlkey=wsc19sue84ytkphheptq28ica&st=9p8npaly&dl=0)
1. Go to the "Side Bar", Click on "Data"
1. Upload as `ecti2021` the file: `ecti2021.zip`  which contains the following four files: `train.zip`, `val_without_ref_labels.zip` , and the `water_tiles.csv` and `background_tiles.csv` from the `data_preparation.ipynb` notebook. 

Add more data: 
1. Downloading the [val_with_ref_labels.zip](https://www.dropbox.com/scl/fi/3w2uvms6lfye9trszgcpz/val_with_ref_labels.zip?rlkey=yz854hfq9uz3nwi51w3ay67d5&st=lnj54acm&dl=0) which was released only at the end of Phase 1 of the ECT1 challenge.
1. Go to the Left "Side Bar", Click on "Home" and "Dataset"
1. Click on the `ecti2021`
1. Click on the three dots in the top-right corner and click on "New Version"
1. Upload the file: `val_with_ref_labels.zip`


Regarding the model you can use any model from Lab_02_model_training_assigment. Otherwise:
1. Download the [model.zip](https://www.dropbox.com/scl/fi/onvm7jkxgerb5ly3e1vfq/models.zip?rlkey=k4aku1wjhpi4tlh4ozf9rhmll&st=w4qdstyw&dl=0)
1. Upload as `models` the following file: `models.zip` or one of the model from Lab02_model_training_assigment (we are conducting error analysis, so the model doesn't need to be perfect at this stage).

# Step 0: Enviroment setting

In [ ]:
!pip install segmentation-models-pytorch
!pip install -U git+https://github.com/albu/albumentations --no-cache-dir

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp  # high-level library of pretrained segmentation models
from sklearn.metrics import confusion_matrix
import pickle
from pickle import load

%matplotlib inline

# Step 1: Load the dataset files

In [ ]:
# Set path to where dataset is downloaded
dataset_root = '/kaggle/input/ecti2021' #set accordingly based on how you uploaded the data
# get number of training/validation regions
test_dir = os.path.join(dataset_root, 'val_with_ref_labels/val_with_ref_labels/test')

n_test_regions = len(glob(test_dir+'/*/'))

# NOTE: make sure number of regions is NOT 0, otherwise it might be that the code is not able to read the data. 
print('Number of test temporal-regions: {}'.format(n_test_regions))

## Utils functions

In [ ]:
def s1_to_rgb(vv_image, vh_image):
    """Convert Sentinel-1 VV/VH bands into a 3-channel false-colour image.

    Channel mapping:
        R = VV   (surface roughness / soil moisture)
        G = VH   (vegetation / volume scattering)
        B = VV / (VH + ε)   (ratio channel, highlights open-water areas)

    Note: the ratio channel is different from Lab-03 which used VV−VH.
    Both are valid composites; the ratio is more sensitive to contrast between
    VV and VH, which helps distinguish flooded vegetation from open water.

    Args:
        vv_image (np.ndarray): VV band, values in [0, 1].
        vh_image (np.ndarray): VH band, values in [0, 1].

    Returns:
        np.ndarray: H×W×3 float array in [0, 1].
    """
    eps = 1e-6  # prevent division by zero over dark (near-zero) pixels
    ratio_image = np.clip(np.nan_to_num(vv_image / (vh_image + eps), nan=0.0), 0, 1)
    rgb_image = np.stack((vv_image, vh_image, ratio_image), axis=2)
    return rgb_image


def get_filename(filepath):
    """Return the filename component of a path (cross-platform)."""
    return os.path.basename(filepath)


def read_csv(csvpath, split_symbol='\\'):
    """Read a text file of paths and return just the filenames.

    This helper is provided for convenience when working with the ETCI
    dataset CSV files that list full Windows-style paths.

    Args:
        csvpath (str): path to the whitespace-delimited file.
        split_symbol (str): path separator used inside the file.

    Returns:
        list[str]: list of bare filenames.
    """
    path_list = np.loadtxt(csvpath, delimiter=" ", dtype=str).tolist()
    return [get_filename(pth) for pth in path_list]

# Step 2: Setup the dataset for machine learning

### Create a PyTorch test dataset
We will be using the PyTorch deep learning library to format this dataset and create our machine learning model. Therefore we will need to create a custom Dataset class and pass it into a DataLoader object (see the [PyTorch Dataset Tutorial](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)  for more detail on the topic). Let's create Dataset and DataLoader objects for the test set. This time we will have labels.

In [ ]:
class ETCIDataset(Dataset):
    def __init__(self, dataframe, split, transform=None):
        self.split = split
        self.dataset = dataframe
        self.transform = transform

    def __len__(self):
        return self.dataset.shape[0]

    def __getitem__(self, index):
        example = {}
        df_row = self.dataset.iloc[index]
        example['region'] = df_row['region']

        # Load VV and VH SAR bands as grayscale, normalised to [0, 1]
        vv_image = cv2.imread(df_row['vv_image_path'], 0) / 255.0
        vh_image = cv2.imread(df_row['vh_image_path'], 0) / 255.0
        assert vv_image is not None, f"Failed to load VV image: {df_row['vv_image_path']}"
        assert vh_image is not None, f"Failed to load VH image: {df_row['vh_image_path']}"

        # Build the false-colour RGB composite (H×W×C, values in [0, 1])
        rgb_image = s1_to_rgb(vv_image, vh_image)

        if self.split == 'test':
            # No flood labels available for the test split
            pass
        else:
            # Load the binary flood mask: pixel value 255 → 1 (flood), 0 → 0 (no flood)
            flood_mask = cv2.imread(df_row['flood_label_path'], 0) / 255.0

            if self.transform:
                # Apply augmentations in H×W×C format before transposing
                augmented = self.transform(image=rgb_image, mask=flood_mask)
                rgb_image = augmented['image']
                flood_mask = augmented['mask']

            example['mask'] = flood_mask.astype('int64')

        # Convert H×W×C → C×H×W (PyTorch expects channels-first)
        example['image'] = rgb_image.transpose((2, 0, 1)).astype('float32')
        return example

In [ ]:
vv_image_paths = sorted(glob(test_dir+'/**/vv/*.png', recursive=True))
vv_image_names = [get_filename(pth) for pth in vv_image_paths]
region_name_dates = ['_'.join(n.split('_')[:2]) for n in vv_image_names]

vh_image_paths, flood_label_paths, water_body_label_paths, region_names = [], [], [], []
for i in range(len(vv_image_paths)):
    # get vh image path
    vh_image_name = vv_image_names[i].replace('vv', 'vh')
    vh_image_path = os.path.join(test_dir, region_name_dates[i], 'tiles', 'vh', vh_image_name)
    vh_image_paths.append(vh_image_path)

    # get flood mask path ()
    flood_label_name = vv_image_names[i].replace('_vv', '')
    flood_label_path = os.path.join(test_dir, region_name_dates[i], 'tiles', 'flood_label', flood_label_name)
    flood_label_paths.append(flood_label_path)
    
    # get water body mask path
    water_body_label_name = vv_image_names[i].replace('_vv', '')
    water_body_label_path = os.path.join(test_dir, region_name_dates[i], 'tiles', 'water_body_label', water_body_label_name)
    water_body_label_paths.append(water_body_label_path)

    # get region name
    region_name = region_name_dates[i]#.split('_')[0] -> this time we want to know all the event data as they are all in florence
    region_names.append(region_name) 

test_paths = {'vv_image_path': vv_image_paths,
        'vh_image_path': vh_image_paths,
        'flood_label_path': flood_label_paths,
        'water_body_label_path': water_body_label_paths,
        'region': region_names
}

test_df = pd.DataFrame(test_paths)

print(test_df.shape)
test_df.head()

# Step 3: Inference

**1. Write a function called `create_model` to load the model**

In [ ]:
device = 'cuda'

def create_model():
    ###CODE HERE
    ###END CODE HERE
    return model

In [ ]:
# Load model weights
model_test = create_model()
# CHANGE the path below to point to your uploaded model checkpoint
model_test.load_state_dict(torch.load('/kaggle/input/models/model.pt'))
model_test.to(device)

batch_size = 64
test_dataset = ETCIDataset(test_df, split='test', transform=None)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,      # must be False to preserve tile order for reconstruction
    num_workers=0,      # 0 avoids DataLoader deadlock on Kaggle
    pin_memory=True,    # speeds up CPU→GPU memory transfers
)

**2. Write a the inference pipeline to generate a `numpy.ndarray` final_predictions of size (10400, 256, 256).**

In [ ]:
final_predictions = []
model_test.eval()
#CODE HERE

#END CODE HERE
final_predictions = np.concatenate(final_predictions, axis=0)

# check final prediction shape
print(final_predictions.shape)

### Visualize some results

***3. Write a function `visualize_results_with_labels` that plot the vv_image,vh_image, prediction, flood_label, and water label***

In [ ]:
def visualize_results_with_labels(df_row, prediction, figsize=(25, 5)):
    """Plot SAR composite, prediction, flood label, water-body label, and combined label.

    When ground-truth labels are available (5-panel view):
        1. SAR RGB composite
        2. Model prediction
        3. Flood label  (new flood water only)
        4. Water body label  (permanent water — rivers, lakes)
        5. Combined label  (flood ∪ water body) — what the model is evaluated against

    When no labels are available (2-panel view):
        1. SAR RGB composite
        2. Model prediction
    """
    

In [ ]:
index = 256
visualize_results_with_labels(test_df.iloc[index], final_predictions[index], figsize=(17,10))

In [ ]:
index = 100
visualize_results_with_labels(test_df.iloc[index], final_predictions[index], figsize=(17,10))

In [ ]:
index = 30
visualize_results_with_labels(test_df.iloc[index], final_predictions[index], figsize=(17,10))

**Note: the above is not a problem as the model was trained on the flood mask and not the water body mask.**

# Step 4: Error Analysis

**4. Calculate the confusion metric of the `prediction` and `df_row` for different indexes using the functions: [sklearn.metrics.confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html). Is the confusion metric outputting always the same number of outputs? The function `.flatten()` and `.ravel()` might be useful.**

In [ ]:
index=25 #change the index
prediction=final_predictions[index]
df_row = test_df.iloc[index]
prediction=prediction.flatten()
flood_label_path = df_row['flood_label_path']
flood_label_image = cv2.imread(flood_label_path, 0) / 255.0
flood_label_image=flood_label_image.astype(int).flatten() 
region = df_row['region']

#CODE HERE

###END CODE HERE

**Why does `confusion_matrix` sometimes return only one value?**


**5. Explain the function `evaluate`, testing it with different indexes.**

In [ ]:
def calc_prec_rec_f1(stat_array):
    """Compute precision, recall, and F1 from [TP, FP, TN, FN]."""
    epsilon = 1e-4
    TP, FP, TN, FN = stat_array
    precision = TP / (TP + FP + epsilon)
    recall    = TP / (TP + FN + epsilon)
    f1        = (2 * precision * recall) / (precision + recall + epsilon)
    return precision, recall, f1


def evaluate(df_row, prediction):
    """Compute per-tile precision, recall, and F1 against the combined water mask.

    The ground-truth label is the union of the flood mask and the permanent
    water-body mask.  This avoids penalising the model for correctly detecting
    rivers and lakes that were already present before the event.

    Returns:
        tuple: (flood_label_path, region, precision, recall, f1)
    """
    prediction = prediction.flatten()

    # Load flood mask and permanent water-body mask, combine with logical OR
    flood_label    = cv2.imread(df_row['flood_label_path'],        0) / 255.0
    water_body     = cv2.imread(df_row['water_body_label_path'],   0) / 255.0
    combined_label = np.logical_or(flood_label > 0, water_body > 0).astype(int).flatten()

    region = df_row['region']

    pred_TN = pred_FP = pred_FN = pred_TP = 0

    try:
        pred_TN, pred_FP, pred_FN, pred_TP = confusion_matrix(combined_label, prediction).ravel()
        assert (pred_FP + pred_TP + pred_FN + pred_TN) == combined_label.size
    except ValueError:
        cm_val = confusion_matrix(combined_label, prediction).ravel()[0]
        if np.all(prediction == 0) and np.all(combined_label == 0):
            pred_TN = cm_val  # pure background tile
        elif np.all(prediction == 1) and np.all(combined_label == 1):
            pred_TP = cm_val  # fully flooded tile
        else:
            print(f'Unexpected confusion matrix shape for: {df_row["flood_label_path"]}')

    stat_array = [pred_TP, pred_FP, pred_TN, pred_FN]
    precision, recall, f1 = calc_prec_rec_f1(stat_array)
    return df_row['flood_label_path'], region, precision, recall, f1

In [ ]:
index=100 #change indexes
prediction=final_predictions[index]
df_row = test_df.iloc[index]
flood_label_path, region, precision, recall,f1=evaluate(df_row,prediction)
print("region",region)
print("flood_label_path", flood_label_path)
print("Precision:",precision)
print("Recal:",recall)
print("F1 score",f1)

In [ ]:
results_list = []
for index in tqdm(range(final_predictions.shape[0])):
    prediction = final_predictions[index]
    df_row = test_df.iloc[index]
    flood_label_path, region, precision, recall, f1 = evaluate(df_row, prediction)
    results_list.append({
        'flood_label_path': flood_label_path,
        'region':           region,
        'precision':        precision,
        'recall':           recall,
        'f1_score':         f1,
    })

**6. Create a [Pandas Data Frame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) to store all the variables:  `flood_label_paths_dict`,`regions_dict`,`prec_dict`,`recall_dict`,and `f1_dict` sorting them by ascending f1 score.**

In [ ]:
###CODE HERE


**7. Use the function [Pandas Pivot Table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html) to display the `region` ordered by `f1 score`.**

In [ ]:
###CODE HERE

###END CODE HERE
print(pivot_table)

**8. Filter out the `f1 score`=0, and add a column with the count of each `region` to identify the margin of improvement. Add plotting as required to better diagnose the model's limitation.**

In [ ]:
###CODE HERE

###END CODE HERE
print(ordered_table)

In [ ]:
# ── Change these variables to explore different regions / tile counts ──────────
worst_region = ordered_table.index[-1]  # ordered_table sorted descending → last = worst
n_to_show    = 10   # None = all tiles, or set e.g. 5 to cap the output
min_f1       = 0.1   # exclude near-empty tiles whose F1 is inflated by epsilon

# ── Find and plot ──────────────────────────────────────────────────────────────
print(f"Region: {worst_region}")
print(f"Mean F1: {ordered_table.loc[worst_region, 'mean_f1_score']:.3f}  |  "
      f"Flood tiles: {ordered_table.loc[worst_region, 'count']}")

worst_tiles = (
    filtered_df[
        (filtered_df['region']   == worst_region) &
        (filtered_df['f1_score'] >= min_f1)
    ]
    .sort_values('f1_score', ascending=True)
)
print(f"Tiles with F1 ≥ {min_f1}: {len(worst_tiles)}")

tiles_to_plot = worst_tiles if n_to_show is None else worst_tiles.head(n_to_show)

for idx, row in tiles_to_plot.iterrows():
    print(f"\n[{idx}]  F1: {row['f1_score']:.3f}  |  "
          f"Precision: {row['precision']:.3f}  |  Recall: {row['recall']:.3f}")
    visualize_results_with_labels(test_df.iloc[idx], final_predictions[idx], figsize=(25, 5))

**9. Come up with a strategy to improve the performances of the model.**

Priorization strategy from Lecture 4